# Budgets: 세션이 지출할 수 있는 금액에 상한 설정하기

사람이 지켜보지 않는 agent에는 비용 측면의 자연스러운 종료 지점이 없습니다. web search를 수행하는 조사 작업은 어느새 fetch를 백 번 더 수행할 수 있습니다. 코딩 루프는 불안정한 테스트를 아침까지 재시도할 수 있습니다. 상한이 없으면 청구서를 받고 나서야 알게 됩니다.

세션 budget은 단일 세션에 강제로 적용되는 지출 상한입니다. 세션을 생성할 때 최대 정가 비용을 설정합니다. 플랫폼은 해당 세션의 모든 thread에서 발생하는 token 비용을 공개 정가로 추적하며, 누적 합계가 상한에 도달하면 세션이 `budget_reached`라는 종료 사유와 함께 idle 상태가 됩니다. 파일, tool 상태, 대화는 그대로 유지됩니다. 상한을 올리면 작업이 멈췄던 지점부터 계속됩니다.

이 notebook에서는 다음을 살펴봅니다.

- `sessions.create`에서 `budget` 설정하기
- `usage.list_cost`와 turn이 끝날 때 stream이 전달하는 `session.usage` snapshot 읽기
- `budget_reached`에 의한 중단과 일시 정지된 세션에 보존되는 항목
- `sessions.update`로 상한을 올리고, 내리고, 제거하기


## 1. client 설정하기

Budgets는 표준 `managed-agents-2026-04-01` beta header를 사용합니다. `budget` parameter와 `session.usage`의 비용 필드를 사용하려면 `anthropic>=0.121.0`이 필요합니다.


In [1]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [2]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
client = anthropic.Anthropic()

## 2. 과다 지출할 여지가 있는 agent 만들기

이 예제의 agent는 경쟁 구도 분석 보고서를 작성합니다. web search와 web fetch를 사용할 수 있고, prompt에는 읽을 출처 수를 제한하는 내용이 없습니다. budget은 바로 이런 개방적인 작업을 위한 것입니다.


In [3]:
env = client.beta.environments.create(
    name="budget-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

analyst = client.beta.agents.create(
    name="market_analyst",
    description="Writes sourced competitive landscape briefs.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-cap-session-spend"},
    model={"id": MODEL},
    system="""You write competitive landscape briefs for product teams.
Given a market, use web search and web fetch to find the notable vendors, their
positioning, and recent moves. Read primary sources rather than aggregators.
Write the brief to /mnt/session/outputs/brief.md with a Sources section listing every
URL you relied on. Keep researching until you are confident the brief is complete.""",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
    betas=BETAS,
)
print(f"{analyst.name}: {analyst.id} v{analyst.version}")

market_analyst: agent_staging_014U3BWzuiSNDwfzubN38hnz v1


## 3. budget을 설정하여 세션 생성하기

`sessions.create`의 `budget` 필드로 상한을 설정합니다.

```json
{"type": "limit", "max_list_cost": {"currency": "USD", "amount": "10"}}
```

금액을 정하기 전에 알아 두면 좋은 몇 가지 특성이 있습니다.

- `amount`는 통화의 최소 단위로 표현한 정수 문자열입니다. `"50"`은 50센트이고, `"2500"`은 $25.00입니다. 문자열로 유지되며 1센트 미만의 금액은 거부되므로 부동소수점 반올림은 적용되지 않습니다. 허용되는 통화는 `USD`뿐이며 API가 반환하는 모든 비용 금액(`usage.list_cost` 포함)은 같은 인코딩을 사용합니다.
- 상한에는 subagent thread를 포함하여 세션의 모든 thread에서 발생한 model token 비용을 공개 정가로 합산합니다. 협의된 할인과 관계없이 정가가 적용되므로 실제 청구 금액이 상한에 도달하기 전에, 또는 도달하는 시점에 상한이 작동합니다. `session.usage`와 공개 요금표로 해당 금액을 재현할 수 있습니다.
- 세션에서 실행할 수 있는 모든 model에는 공개 정가가 있어야 합니다. 그렇지 않은 model이 있으면 생성이 `model_not_budgetable`로 실패합니다.
- `budget`을 생략하면 세션에는 상한이 없습니다. budget은 생성 시에만 연결할 수 있습니다. budget 없이 생성된 세션에는 나중에 budget을 추가할 수 없으므로 미리 결정하세요.

여기서는 몇 분 안에 중단을 확인할 수 있도록 상한을 의도적으로 낮은 10센트로 설정합니다. 아래의 모든 출력에서 `usd` helper는 최소 단위 금액을 달러로 표시합니다.


In [4]:
from decimal import Decimal


def usd(money) -> str:
    """정수 최소 단위 금액("50" = 50센트)을 달러로 표시합니다."""
    return f"${Decimal(money.amount) / 100:.2f}"


session = client.beta.sessions.create(
    agent=analyst.id,
    environment_id=env.id,
    title="Landscape brief: observability platforms",
    budget={
        "type": "limit",
        "max_list_cost": {"currency": "USD", "amount": "10"},
    },
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Write a competitive landscape brief on the observability "
                    "platform market: the main vendors, how each positions itself, "
                    "and any notable moves in the last year.",
                }
            ],
        }
    ],
    betas=BETAS,
)
print(session.id, session.status)
print("budget:", usd(session.budget.max_list_cost))

sesn_staging_01ELJgA2vqTR6EmjX8viwHZi running
budget: $0.10


## 4. 지출이 누적되는 과정 확인하기

일반적인 `agent.message` 및 `agent.tool_use` event와 함께, stream은 세션이 `idle` 상태로 들어갈 때마다 `session.usage` snapshot을 전달합니다. 여기에는 누적 token 수, 추적 중인 `list_cost`, 설정된 `budget`이 담기므로 turn이 끝나는 순간 `sessions.retrieve`를 다시 호출하지 않고 비용을 알 수 있습니다.

루프는 `session.status_idle`에서 종료됩니다. budget이 설정되어 있으면 turn을 끝내는 종료 사유는 두 가지입니다. agent가 완료했을 때는 `end_turn`, 상한이 먼저 작동했을 때는 `budget_reached`입니다.


In [5]:
stop_reason = None

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            print(f"[{ev.name}] {str(ev.input)[:80]}")
        elif ev.type == "session.usage":
            print(f"  usage: {usd(ev.usage.list_cost)} of {usd(ev.budget.max_list_cost)}")
        elif ev.type == "session.status_idle":
            stop_reason = ev.stop_reason.type
            print(f"[idle] stop_reason={stop_reason}")
            break

assert stop_reason == "budget_reached", (
    f"expected budget_reached, got {stop_reason}: lower the cap in step 3 "
    "or give the agent a bigger task"
)

[repl] {'script': '\nlet queries = [\n  "Datadog positioning observability platform 202
[web_search] {'query': 'Datadog positioning observability platform 2025'}
[web_search] {'query': 'New Relic observability platform positioning 2025'}
[web_search] {'query': 'Dynatrace observability platform AI 2025'}
[web_search] {'query': 'Splunk Cisco observability 2025'}
[web_search] {'query': 'Grafana Labs observability platform 2025 announcement'}
[web_search] {'query': 'Elastic observability 2025'}
[web_search] {'query': 'Honeycomb observability 2025'}
[web_search] {'query': 'Chronosphere observability 2025 acquisition'}
[web_search] {'query': 'Datadog acquisition 2024 2025'}
[web_search] {'query': 'Grafana Labs acquisition 2024 2025'}


  usage: $0.12 of $0.10
[idle] stop_reason=budget_reached


## 5. 일시 정지된 세션 확인하기

`budget_reached`는 실패가 아니라 일시 정지입니다. 세션은 `idle`에 머무르고, `usage`에는 지금까지 지출한 모든 금액이 반영되며, container에는 상한에 도달하기 전에 agent가 작성한 내용이 그대로 남습니다. 다시 실행할 필요가 없습니다.

상한은 model 요청 사이에서 적용되므로 기록된 `list_cost`가 `max_list_cost`를 조금 초과할 수 있습니다. 한도를 넘기는 요청이 완료된 다음 세션이 일시 정지됩니다. 이 요청 한 번의 여유분을 고려하여 상한을 정하세요.


In [6]:
paused = client.beta.sessions.retrieve(session.id, betas=BETAS)

print("status:      ", paused.status)
print("list cost:   ", usd(paused.usage.list_cost))
print("cap:         ", usd(paused.budget.max_list_cost))
print("tokens:       in", paused.usage.input_tokens, "out", paused.usage.output_tokens)
print("active secs: ", round(paused.usage.active_seconds, 1))

status:       idle
list cost:    $0.12
cap:          $0.10
tokens:       in 471 out 606
active secs:  13.7


event log를 보면 agent가 어디까지 진행했는지 알 수 있습니다. 실행한 tool 호출과, 설명을 시작했다면 마지막 message가 나타납니다. 조사를 깊이 진행 중인 agent는 상한이 작동할 때 아직 아무것도 작성하지 않았을 수 있으므로 message는 선택적으로 존재하는 것으로 취급하세요.


In [7]:
events = list(client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS))
tool_calls = [ev for ev in events if ev.type == "agent.tool_use"]
messages = [ev for ev in events if ev.type == "agent.message"]

print(f"tool calls before the cap: {len(tool_calls)}")
if messages:
    last = messages[-1]
    print("".join(b.text for b in last.content if b.type == "text")[:600])
else:
    print("(no agent.message yet: the agent was still gathering sources when the cap fired)")

tool calls before the cap: 11
(no agent.message yet: the agent was still gathering sources when the cap fired)


## 6. 상한을 올리고 완료하게 하기

`sessions.update`는 동일한 형태의 `budget`을 받습니다. `max_list_cost`를 이미 소비한 비용보다 높이면 일시 정지가 해제되며, 세션은 중단 당시 상태에서 중단된 turn을 스스로 재개합니다.

상한을 이미 지출한 금액 이하로 내리면 400으로 거부됩니다. `max_list_cost`는 소비한 정가 비용보다 높게 유지되어야 하므로 update로 세션이 자신의 과거 사용량에 막혀 움직이지 못하는 상태가 되는 일은 없습니다. 첫 번째 호출은 이 거부를 보여 주고, 두 번째 호출은 실제로 상한을 올립니다. 상한을 올린 뒤 후속 message는 필요 없습니다. 세션은 다시 `running`으로 전환하고 중단된 turn을 이어갑니다.


In [8]:
try:
    client.beta.sessions.update(
        session.id,
        budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "1"}},
        betas=BETAS,
    )
except anthropic.BadRequestError as e:
    print("lower rejected:", e.message)

client.beta.sessions.update(
    session.id,
    budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "500"}},
    betas=BETAS,
)
print("cap raised to $5.00")

lower rejected: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': "`budget.max_list_cost` must be greater than the session's consumed list cost"}, 'request_id': 'req_staging_011CdiPi8iPJKWdiuurZuDkB'}


cap raised to $5.00


In [9]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.status_running":
            print("[resumed]")
        elif ev.type == "agent.tool_use":
            print(f"[{ev.name}]")
        elif ev.type == "session.status_idle":
            print(f"[idle] stop_reason={ev.stop_reason.type}")
            break

done = client.beta.sessions.retrieve(session.id, betas=BETAS)
print("final list cost:", usd(done.usage.list_cost), "of cap", usd(done.budget.max_list_cost))

[resumed]


[repl]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]


[repl]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]


[repl]
[web_fetch]


[web_fetch]
[web_fetch]


...

[write]


[write]


[bash]


[idle] stop_reason=end_turn


final list cost: $1.83 of cap $5.00


## 7. 보고서 읽기

agent는 context를 그대로 유지한 채 작업을 다시 이어받아 작성을 완료했습니다. event log에서 생성한 파일을 가져옵니다.


In [10]:
brief = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input.get("file_path", "").endswith("brief.md")
    ):
        brief = ev.input["content"]  # 수정이 있을 수 있으므로 마지막 write를 보관합니다

print(brief[:1500] + ("\n..." if len(brief) > 1500 else ""))

# Competitive Landscape Brief: Observability Platform Market

*Scope: commercial platforms for metrics, logs, traces, and related telemetry (APM, infrastructure monitoring, log management, digital experience monitoring) used by engineering, IT operations, and SRE teams. Window for "recent moves": roughly the last twelve months of vendor activity found in primary sources (mid‑2025 through mid‑2026).*

## 1. Market shape

Observability has consolidated into a market with a handful of large, broad "full platform" vendors (Datadog, Splunk/Cisco, Dynatrace, New Relic, Elastic, Grafana Labs), a tier of cost‑ and scale‑focused challengers (Chronosphere, Coralogix, Honeycomb, SolarWinds, LogicMonitor, Sumo Logic), hyperscaler-native tooling bundled into the cloud platforms (AWS CloudWatch, Azure Monitor, Google Cloud Observability), and an open standard — OpenTelemetry — that now sits underneath almost every vendor's ingestion layer and was elevated to CNCF "graduated" status, cementing it as 

## 8. budget 제거하기

update에서 `budget=None`을 전달하면 상한이 해제되어 그 시점부터 세션에는 상한이 없습니다. `budget` 자체를 생략하면 현재 상한이 유지됩니다. 두 요청은 다릅니다. `None`은 명시적인 제거이고, 부재는 보존입니다.

연결이 생성 시에만 가능한 것과 같은 이유로 제거는 영구적입니다. budget이 없어진 세션에는 다시 budget을 추가할 수 없습니다. 상한은 원하는 만큼 올리고 내릴 수 있지만, 제거는 되돌릴 수 없는 결정으로 생각하세요.


In [11]:
uncapped = client.beta.sessions.update(session.id, budget=None, betas=BETAS)
print("budget:", uncapped.budget)

budget: None


## 9. 정리하기


In [12]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")

archived


## budget을 활용할 곳

budget은 세션 단위 제어 수단입니다. 한 번의 실행을 제한하며 조직, workspace 또는 하루의 traffic을 제한하지 않으므로, 해당 수준에 이미 설정한 지출 한도와 함께 작동하고 이를 대체하지 않습니다. 사람이 지켜보지 않는 세션에 사용하세요. cron으로 실행되는 deployment, webhook으로 시작되는 agent, coordinator가 나누는 작업 수가 데이터에 따라 달라지는 장시간 multiagent 작업이 그 예입니다.

streaming하지 않는 대규모 세션 집합에는 연결을 계속 열어 두는 대신 `session.budget_reached` webhook을 구독하세요. 세션이 상한에 도달하면 한 번 발생하며 payload에 세션이 명시되므로, 감독 process가 상한을 올릴지 세션을 일시 정지 상태로 둘지 결정할 수 있습니다. webhook 패턴은 [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb)에서 다룹니다.
